# MERLIN – training notebook (image of a formula → LaTeX)

**MERLIN** = *Math Expression Recognition with Learned INvariance*. The model is an encoder–decoder: DenseNet-121 up to block 3 + 2D positional encoding + coverage attention + a 4-layer GRU decoder with in-model knowledge distillation.

This one notebook trains every variant used in the thesis (`EXPERIMENT`), evaluates it on the **full** im2latex-100k test set, runs the font-robustness test and exports the model for the web app.

| Topic | How the notebook handles it |
|---|---|
| Contribution of each component | **Ablation study** – the same notebook trains the variants (`EXPERIMENT`); cell 8 builds a comparison table |
| Single training run | `SEED` parameter – the same experiment can be repeated with 3 seeds |
| Evaluation on a sample only | Text metrics and image match on the **full test set** (10,355); beam search and early exit on the same 1,000 random formulas |
| Early exit not measured | Early exit is **measured** (accuracy, average number of layers, time) + an accuracy–speed curve; skipped layers receive a **copy of the state** |
| Generation capped at 150 tokens | `MAX_GEN_LEN = 250` |
| Exposure bias | **Scheduled sampling** (up to 20 % of the model's own predictions as input, from epoch 3) |
| KD loss computed on `<PAD>` | Masked |
| Repetition penalty worked only for positive logits | Fixed (divides positive, multiplies negative logits) |
| Tokenizer trained on test formulas too | Trained **only on training** formulas |
| Model fails on other fonts and on real screenshots | **`'robust'` experiment**: the same training formulas rendered in 11–16 LaTeX fonts and 3 matplotlib fonts + augmentation (margins/size, stroke width, blur, rotation, JPEG, noise, contrast). A new **robustness test**: test formulas in training fonts and in fonts the model has never seen |

The model code lives in `merlin_model.py` (cell 2), the font rendering and augmentation code in `merlin_data.py`. **The web app uses the very same file**, so the model on the site is identical to the one evaluated here.

## How to run
1. *Runtime → Change runtime type → T4 GPU*.
2. In cell 1 choose `EXPERIMENT` (start with `'baseline'`) and *Run all*. On the first run cell 3b also builds the **robustness test** (test formulas in other fonts, ≈ 10–15 min, once), so every experiment gets that result too. Training is checkpointed after every epoch – if Colab disconnects, just *Run all* again and it resumes where it stopped. The same holds for evaluation.
3. **Robust model (used by the web app):** `EXPERIMENT = 'robust'`. The first time, cell 3b renders the formulas in many fonts (≈ 30–40 min, once), then training takes ≈ 3–3.5 h and evaluation ≈ 1.5–2 h (including the robustness test for this and the baseline model). Cell 9 then exports this model for the app.
4. For the ablation study repeat with `EXPERIMENT = 'no_aspect'`, `'block4'`, `'pos1d'`, `'no_coverage'`, `'no_ss'` (each ≈ 3 h of training + ≈ 1.5 h of evaluation). Cell 8 collects all finished experiments into one table.
5. Cell 9 exports the `'robust'` model for the web app (and, if `HF_TOKEN` is set, uploads it straight to a Hugging Face Space).

## 1. Setup and experiment selection

In [ ]:
# CELL 1: Setup and experiment selection
from google.colab import drive
import os, json, time, random, math, subprocess
import numpy as np

!pip install -q python-Levenshtein kaggle tokenizers huggingface_hub
if subprocess.run(["which", "pdflatex"], capture_output=True).returncode != 0 or \
   subprocess.run(["which", "pdftoppm"], capture_output=True).returncode != 0:
    print("Installing pdflatex and poppler (2-3 min)...")
    os.system("apt-get update -qq && apt-get install -y -qq texlive-latex-base texlive-latex-extra "
              "texlive-fonts-recommended poppler-utils > /dev/null 2>&1")
drive.mount('/content/drive')

# ------------------------------------------------------------------------------------------
EXPERIMENT = 'baseline'      # 'baseline' | 'robust' | 'no_aspect' | 'block4' | 'pos1d' | 'no_coverage' | 'no_ss'
SEED = 42                  # for repeated runs: 42, 43, 44
# ------------------------------------------------------------------------------------------

BASE = dict(preprocess='aspect', encoder_block=3, pos='2d', coverage=True, ss_max=0.2, synth=False, aug=False)
EXPERIMENTS = {            # ablation: every variant changes EXACTLY one component of the baseline model
    'baseline':       dict(BASE),
    'no_aspect':   dict(BASE, preprocess='square'),
    'block4':        dict(BASE, encoder_block=4),
    'pos1d':        dict(BASE, pos='1d'),
    'no_coverage': dict(BASE, coverage=False),
    'no_ss':       dict(BASE, ss_max=0.0),
    # robust model: baseline + the same formulas in many fonts + augmentation (this one goes to the web app)
    'robust':       dict(BASE, synth=True, aug=True),
}
EXP = EXPERIMENTS[EXPERIMENT]
MODEL_CFG = {k: v for k, v in EXP.items() if k not in ('ss_max', 'synth', 'aug')}

TRAIN = dict(batch_size=64, epochs=10, lr=1e-4, max_lr=6e-4, weight_decay=1e-4,
             kd_T=2.0, kd_alpha=0.4, student_weight=0.2, ss_max=EXP['ss_max'], ss_start_epoch=3)
MAX_GEN_LEN = 250
EVAL = dict(n_subset=1000, subset_seed=42, beam_width=3, ee_curve_n=300,
            ee_thresholds=[0.05, 0.2, 0.5, 1.0, 2.0], full_image_match=True)

SAVE_ROOT = '/content/drive/MyDrive/MERLIN'
RUN_DIR = os.path.join(SAVE_ROOT, 'experiments', f'{EXPERIMENT}_seed{SEED}')
RENDER_DIR = os.path.join(SAVE_ROOT, 'renders_gold')        # cache of rendered ground-truth formulas (shared)
for d in (SAVE_ROOT, RUN_DIR, RENDER_DIR):
    os.makedirs(d, exist_ok=True)

RESULTS_PATH = os.path.join(RUN_DIR, 'results.json')
def load_results():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, encoding='utf-8') as f:
            return json.load(f)
    return {}
def save_results(update):
    R = load_results(); R.update(update)
    with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
        json.dump(R, f, ensure_ascii=False, indent=1)
    return R

save_results({'experiment': EXPERIMENT, 'seed': SEED, 'model_cfg': MODEL_CFG, 'data': {'synth': EXP['synth'], 'aug': EXP['aug']}, 'train': TRAIN, 'eval': EVAL,
              'max_gen_len': MAX_GEN_LEN})
print("Experiment:", EXPERIMENT, MODEL_CFG, "| folder:", RUN_DIR)

## 2. Model code (`merlin_model.py`)
This file is identical to `merlin_model.py` in the repository and in the web app. It contains image preprocessing, the model (with component switches for the ablation), the KD loss and all decoding methods.

In [ ]:
%%writefile /content/merlin_model.py
# -*- coding: utf-8 -*-
"""
merlin_model.py – shared model code for generating LaTeX from an image of a formula.

The same file is used by the Colab notebook (training, evaluation) and by the web app (inference),
so the model in the app is guaranteed to be the model described in the thesis.

Components are selected by the configuration (for the ablation study):
    preprocess    'aspect' (height 96 px, aspect ratio preserved) | 'square' (224×224)
    encoder_block 3 (stride 16, finer feature map) | 4 (stride 32)
    pos           '2d' (row, column) | '1d' (learned vector per region index)
    coverage      True | False
"""
import math
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms.functional as TF
from PIL import Image, ImageOps

DEFAULT_CONFIG = dict(
    preprocess='aspect', img_height=96, max_width=768, square_size=224,
    encoder_block=3, pos='2d', coverage=True,
    embed_size=256, hidden_size=512, num_layers=4,
)
SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def full_config(cfg=None):
    c = dict(DEFAULT_CONFIG)
    c.update(cfg or {})
    return c


def feat_stride(cfg):
    return 16 if cfg['encoder_block'] == 3 else 32


# =========================================================================================
#  Tokenizer
# =========================================================================================
def build_tokenizer(train_formulas_txt, vocab_size=5000):
    """BPE that reproduces the LaTeX source exactly: it splits only on spaces, and the </w> suffix marks
    the end of a LaTeX token. It is trained ONLY on formulas from the training set."""
    from tokenizers import Tokenizer, decoders
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import WhitespaceSplit
    tok = Tokenizer(BPE(unk_token="<UNK>", end_of_word_suffix="</w>"))
    tok.pre_tokenizer = WhitespaceSplit()
    tok.decoder = decoders.BPEDecoder(suffix="</w>")
    tok.train([train_formulas_txt], BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL_TOKENS,
                                               end_of_word_suffix="</w>", show_progress=False))
    return tok


class TokenizerWrapper:
    def __init__(self, tok):
        self.tok = tok
        self.pad, self.sos, self.eos, self.unk = (tok.token_to_id(t) for t in SPECIAL_TOKENS)
        self.vocab_size = tok.get_vocab_size()

    @classmethod
    def from_file(cls, path):
        from tokenizers import Tokenizer
        return cls(Tokenizer.from_file(path))

    def encode(self, formula):
        return [self.sos] + self.tok.encode(formula).ids + [self.eos]

    def decode(self, ids):
        ids = [i for i in ids if i not in (self.pad, self.sos, self.eos)]
        return ' '.join(self.tok.decode(ids).split())

    def piece(self, i):
        """Text of one BPE token and whether it ends a LaTeX token (for per-token display)."""
        s = self.tok.id_to_token(i) or ''
        return s.replace('</w>', ''), s.endswith('</w>')


# =========================================================================================
#  Image preprocessing
# =========================================================================================
class ImagePreprocessor:
    """PIL image -> tensor [1, h, w] in the range 0..1 (1 = white)."""
    def __init__(self, cfg):
        self.cfg = full_config(cfg)

    def __call__(self, image):
        image = image.convert('L')
        if self.cfg['preprocess'] == 'square':
            s = self.cfg['square_size']
            return TF.to_tensor(image.resize((s, s), Image.BILINEAR))
        w, h = image.size
        s = min(self.cfg['img_height'] / h, self.cfg['max_width'] / w)
        image = image.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BILINEAR)
        return TF.to_tensor(image)


def prepare_user_image(image, ink_frac=None, pad_ratio=0.08):
    """User image (screenshot, photo) -> the look of the dataset images:
    greyscale, dark formula on white, cropped to the content with a white margin.

    ink_frac: share of the image height taken by the formula. In im2latex the formulas are surrounded by
    white space, so after scaling to 96 px they occupy only part of the height. A screenshot cropped
    "to the edge" would give much larger glyphs than in training, so white space is added around it
    (the app tries several values and keeps the one the model is most confident in)."""
    if image.mode in ('RGBA', 'LA') or (image.mode == 'P' and 'transparency' in image.info):
        bg = Image.new('RGBA', image.size, (255, 255, 255, 255))
        image = Image.alpha_composite(bg, image.convert('RGBA'))
    g = np.asarray(image.convert('L'), dtype=np.float32)
    # dark background (e.g. dark mode) -> invert; the formula covers a minority of pixels, so the median = background
    if np.median(g) < 128:
        g = 255 - g
    # contrast stretching
    lo, hi = np.percentile(g, 1), np.percentile(g, 99)
    if hi - lo > 10:
        g = np.clip((g - lo) / (hi - lo) * 255, 0, 255)
    # ink/background threshold: Otsu's method (adapts to grey backgrounds and faint ink)
    hist = np.histogram(g, bins=256, range=(0, 256))[0].astype(np.float64)
    w0 = np.cumsum(hist); w1 = w0[-1] - w0
    m = np.cumsum(hist * np.arange(256)); mu0 = m / np.maximum(w0, 1); mu1 = (m[-1] - m) / np.maximum(w1, 1)
    thr = float(np.clip(np.argmax(w0 * w1 * (mu0 - mu1) ** 2), 100, 200))
    # remove frames/shadows along the border (rows and columns that are mostly dark)
    had_frame = False
    for _ in range(4):
        d = g < thr
        changed = False
        while g.shape[0] > 8 and d[0].mean() > 0.5:
            g, d, changed = g[1:], d[1:], True
        while g.shape[0] > 8 and d[-1].mean() > 0.5:
            g, d, changed = g[:-1], d[:-1], True
        while g.shape[1] > 8 and d[:, 0].mean() > 0.5:
            g, d, changed = g[:, 1:], d[:, 1:], True
        while g.shape[1] > 8 and d[:, -1].mean() > 0.5:
            g, d, changed = g[:, :-1], d[:, :-1], True
        had_frame |= changed
        if not changed:
            break
    if had_frame:                      # frame remnants (rounded corners) at the very edge -> white
        k = max(3, int(round(0.05 * min(g.shape))))
        g = g.copy()
        g[:k, :] = 255; g[-k:, :] = 255; g[:, :k] = 255; g[:, -k:] = 255
    ink = g < thr
    if ink.any():
        rows, cols = np.where(ink.any(1))[0], np.where(ink.any(0))[0]
        g = g[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]
    h = g.shape[0]
    side = max(4, int(round(pad_ratio * h)))
    if ink_frac:
        v = max(side, int(round(h * (1.0 / ink_frac - 1.0) / 2)))
        g = np.pad(g, ((v, v), (side, side)), constant_values=255)
    else:
        g = np.pad(g, side, constant_values=255)
    return Image.fromarray(g.astype(np.uint8))


def collate_images(images, cfg):
    """List of tensors [1, h, w] -> (batch [B, 3, H, W] normalised for DenseNet, mask [B, H'*W']).
    Images are placed on a white canvas; the mask marks which regions are real image (not white padding)."""
    cfg = full_config(cfg)
    stride = feat_stride(cfg)
    H = math.ceil(max(im.shape[1] for im in images) / stride) * stride
    W = math.ceil(max(im.shape[2] for im in images) / stride) * stride
    canvas = torch.ones(len(images), 1, H, W)
    valid_cols = []
    for b, im in enumerate(images):
        h, w = im.shape[1:]
        top = (H - h) // 2
        canvas[b, :, top:top + h, :w] = im
        valid_cols.append(math.ceil(w / stride))
    batch = (canvas.repeat(1, 3, 1, 1) - IMAGENET_MEAN) / IMAGENET_STD
    Hf, Wf = H // stride, W // stride
    cols = torch.arange(Wf).view(1, 1, Wf).expand(len(images), Hf, Wf)
    mask = (cols < torch.tensor(valid_cols).view(-1, 1, 1)).reshape(len(images), -1)
    return batch, mask


# =========================================================================================
#  Model
# =========================================================================================
class PositionalEncoding2D(nn.Module):
    """Sinusoidal 2D encoding: half of the channels encode the row (y), half the column (x)."""
    def __init__(self, d_model, max_h=64, max_w=512):
        super().__init__()
        d = d_model // 2
        self.register_buffer('pe_y', self._pe_1d(max_h, d), persistent=False)
        self.register_buffer('pe_x', self._pe_1d(max_w, d), persistent=False)

    @staticmethod
    def _pe_1d(n, d):
        pos = torch.arange(n).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe = torch.zeros(n, d)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        return pe

    def forward(self, x):                                    # x: [B, C, H, W]
        _, _, H, W = x.shape
        pe = torch.cat([self.pe_y[:H].unsqueeze(1).expand(H, W, -1),
                        self.pe_x[:W].unsqueeze(0).expand(H, W, -1)], dim=-1)
        return x + pe.permute(2, 0, 1).unsqueeze(0).to(x.dtype)


class DenseNetEncoder(nn.Module):
    def __init__(self, cfg, pretrained=True):
        super().__init__()
        self.cfg = cfg
        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        f = models.densenet121(weights=weights).features
        layers = [f.conv0, f.norm0, f.relu0, f.pool0, f.denseblock1, f.transition1,
                  f.denseblock2, f.transition2, f.denseblock3]
        if cfg['encoder_block'] == 3:
            self.norm = f.transition3.norm                   # output of block 3 (stride 16)
        else:
            layers += [f.transition3, f.denseblock4]
            self.norm = f.norm5                              # output of block 4 (stride 32)
        self.features = nn.Sequential(*layers)
        E = cfg['embed_size']
        self.conv = nn.Conv2d(1024, E, kernel_size=1)
        if cfg['pos'] == '2d':
            self.pos2d = PositionalEncoding2D(E)
        else:
            self.pos1d = nn.Parameter(torch.randn(1, 2048, E) * 0.02)

    def forward(self, images):
        x = F.relu(self.norm(self.features(images)))
        x = F.relu(self.conv(x))
        if self.cfg['pos'] == '2d':
            x = self.pos2d(x)
        _, _, H, W = x.shape
        x = x.flatten(2).permute(0, 2, 1)                    # [B, H*W, E]
        if self.cfg['pos'] == '1d':
            x = x + self.pos1d[:, :x.size(1), :].to(x.dtype)
        return x, (H, W)


class CoverageAttention(nn.Module):
    def __init__(self, hidden_size, embed_size, use_coverage=True, cov_channels=32, cov_kernel=(3, 7)):
        super().__init__()
        self.use_coverage = use_coverage
        self.q_linear = nn.Linear(hidden_size, hidden_size)
        self.kv_linear = nn.Linear(embed_size, hidden_size * 2)
        self.scale = hidden_size ** -0.5
        if use_coverage:
            self.cov_conv = nn.Conv2d(1, cov_channels, cov_kernel, padding=(cov_kernel[0] // 2, cov_kernel[1] // 2))
            self.cov_proj = nn.Linear(cov_channels, hidden_size // 4)
            self.cov_score = nn.Linear(hidden_size // 4, 1, bias=False)
            nn.init.zeros_(self.cov_score.weight)

    def precompute(self, features):
        key, value = self.kv_linear(features).chunk(2, dim=-1)
        return key, value

    def forward(self, hidden, key, value, coverage, hw, mask=None):
        q = self.q_linear(hidden).unsqueeze(1)                                   # [B, 1, Hd]
        scores = (torch.bmm(q, key.transpose(1, 2)).squeeze(1) * self.scale).float()   # [B, N]
        if self.use_coverage:
            H, W = hw
            cov = self.cov_conv(coverage.view(-1, 1, H, W).to(q.dtype)).flatten(2).transpose(1, 2)
            scores = scores + self.cov_score(torch.tanh(self.cov_proj(cov))).squeeze(-1).float()
        if mask is not None:
            scores = scores.masked_fill(~mask, float('-inf'))
        alpha = F.softmax(scores, dim=-1)
        context = torch.bmm(alpha.unsqueeze(1).to(value.dtype), value).squeeze(1)
        return context, alpha


class GRUDecoder(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        E, Hd, L = cfg['embed_size'], cfg['hidden_size'], cfg['num_layers']
        self.num_layers = L
        self.embedding = nn.Embedding(vocab_size, E)
        self.attention = CoverageAttention(Hd, E, cfg['coverage'])
        self.init_hidden = nn.Linear(E, Hd)
        self.gru_layers = nn.ModuleList([nn.GRUCell(E + Hd if i == 0 else Hd, Hd) for i in range(L)])
        self.classifier = nn.Linear(Hd, vocab_size)

    def init_state(self, features, mask):
        m = mask.unsqueeze(-1).to(features.dtype)
        h0 = self.init_hidden((features * m).sum(1) / m.sum(1).clamp(min=1))
        coverage = torch.zeros(features.size(0), features.size(1), device=features.device)
        return [h0] * self.num_layers, coverage

    def forward(self, features, hw, mask, captions, ss_prob=0.0):
        """Training pass. ss_prob > 0 enables scheduled sampling: with that probability the input token
        is the model's prediction from the previous step instead of the ground-truth token."""
        key, value = self.attention.precompute(features)
        hidden, coverage = self.init_state(features, mask)
        outs = {i: [] for i in range(self.num_layers)}
        prev_pred = None
        for t in range(captions.size(1)):
            token = captions[:, t]
            if ss_prob > 0 and prev_pred is not None:
                use = torch.rand(token.size(0), device=token.device) < ss_prob
                token = torch.where(use, prev_pred, token)
            context, alpha = self.attention(hidden[-1], key, value, coverage, hw, mask)
            coverage = coverage + alpha
            x = torch.cat([self.embedding(token), context], dim=1)
            for i, cell in enumerate(self.gru_layers):
                hidden[i] = cell(x, hidden[i])
                x = hidden[i]
                outs[i].append(self.classifier(x))
            prev_pred = outs[self.num_layers - 1][-1].detach().argmax(-1)
        return {i: torch.stack(v, dim=1) for i, v in outs.items()}


class Im2LatexModel(nn.Module):
    def __init__(self, cfg, vocab_size, pretrained=True):
        super().__init__()
        self.cfg = full_config(cfg)
        self.encoder = DenseNetEncoder(self.cfg, pretrained)
        self.decoder = GRUDecoder(self.cfg, vocab_size)

    def forward(self, images, mask, captions, ss_prob=0.0):
        features, hw = self.encoder(images)
        return self.decoder(features, hw, mask, captions, ss_prob)


class KDLoss(nn.Module):
    """CE with the ground-truth token + KL towards the teacher's softened distribution; <PAD> positions are ignored."""
    def __init__(self, temperature=2.0, alpha=0.4, pad_idx=0):
        super().__init__()
        self.T, self.alpha, self.pad_idx = temperature, alpha, pad_idx

    def forward(self, student_logits, teacher_logits, targets):
        V = student_logits.size(-1)
        y = targets.reshape(-1)
        valid = y != self.pad_idx
        s = student_logits.reshape(-1, V)[valid].float()
        t = teacher_logits.reshape(-1, V)[valid].float()
        loss_ce = F.cross_entropy(s, y[valid])
        loss_kd = F.kl_div(F.log_softmax(s / self.T, -1), F.softmax(t / self.T, -1),
                           reduction='batchmean') * self.T ** 2
        return (1.0 - self.alpha) * loss_ce + self.alpha * loss_kd


def total_loss(model, outputs, targets, pad_idx, kd_loss, student_weight=0.2):
    L = model.decoder.num_layers
    V = outputs[L - 1].size(-1)
    teacher = outputs[L - 1].detach()
    ce_top = F.cross_entropy(outputs[L - 1].reshape(-1, V).float(), targets.reshape(-1), ignore_index=pad_idx)
    loss = ce_top
    for i in range(L - 1):
        loss = loss + student_weight * kd_loss(outputs[i], teacher, targets)
    return loss, ce_top


# =========================================================================================
#  Generation
# =========================================================================================
def _step(dec, token, hidden, key, value, coverage, hw, mask):
    context, alpha = dec.attention(hidden[-1], key, value, coverage, hw, mask)
    x = torch.cat([dec.embedding(token), context], dim=1)
    new_hidden = []
    for i, cell in enumerate(dec.gru_layers):
        x = cell(x, hidden[i])
        new_hidden.append(x)
    return dec.classifier(x), new_hidden, coverage + alpha, alpha


def _cut(seq, eos):
    out = []
    for t in seq:
        if t == eos:
            break
        out.append(t)
    return out


@torch.no_grad()
def greedy_batch(model, images, mask, sos, eos, max_len=250):
    """Greedy decoding through all 4 layers for a whole batch at once (fast evaluation of the full test set)."""
    model.eval()
    dec = model.decoder
    features, hw = model.encoder(images)
    key, value = dec.attention.precompute(features)
    hidden, coverage = dec.init_state(features, mask)
    B = images.size(0)
    token = torch.full((B,), sos, dtype=torch.long, device=images.device)
    out = torch.full((B, max_len), eos, dtype=torch.long, device=images.device)
    done = torch.zeros(B, dtype=torch.bool, device=images.device)
    for t in range(max_len):
        logits, hidden, coverage, _ = _step(dec, token, hidden, key, value, coverage, hw, mask)
        token = logits.argmax(-1)
        out[:, t] = torch.where(done, torch.full_like(token, eos), token)
        done |= token == eos
        if done.all():
            break
    return [_cut(s, eos) for s in out.tolist()]


@torch.no_grad()
def beam_search(model, image, mask, sos, eos, beam_width=3, max_len=250):
    """Beam search for one image; all beams are computed together as a batch, each with its own
    coverage map. The sequence with the best length-normalised log-probability is returned."""
    model.eval()
    dec, k = model.decoder, beam_width
    features, hw = model.encoder(image)
    key, value = dec.attention.precompute(features)
    hidden, coverage = dec.init_state(features, mask)
    key, value, mask_k = key.expand(k, -1, -1), value.expand(k, -1, -1), mask.expand(k, -1)
    hidden = [h.expand(k, -1).contiguous() for h in hidden]
    coverage = coverage.expand(k, -1).contiguous()
    seqs = [[sos] for _ in range(k)]
    scores = torch.full((k,), float('-inf'), device=image.device)
    scores[0] = 0.0
    completed = []
    for _ in range(max_len):
        token = torch.tensor([s[-1] for s in seqs], device=image.device)
        logits, new_hidden, new_cov, _ = _step(dec, token, hidden, key, value, coverage, hw, mask_k)
        logp = F.log_softmax(logits.float(), dim=-1)
        V = logp.size(1)
        top_s, top_i = (scores.unsqueeze(1) + logp).view(-1).topk(2 * k)
        nxt = []
        for s, idx in zip(top_s.tolist(), top_i.tolist()):
            if s == float('-inf'):
                continue
            b, w = divmod(idx, V)
            if w == eos:
                completed.append((s / (len(seqs[b]) + 1), seqs[b][1:]))
            elif len(nxt) < k:
                nxt.append((s, b, w))
        if len(completed) >= k or not nxt:
            break
        sel = torch.tensor([b for _, b, _ in nxt], device=image.device)
        seqs = [seqs[b] + [w] for _, b, w in nxt]
        scores = torch.tensor([s for s, _, _ in nxt], device=image.device)
        hidden = [h[sel] for h in new_hidden]
        coverage = new_cov[sel]
        n = len(nxt)
        key, value, mask_k = key[:n], value[:n], mask_k[:n]
    if not completed:
        completed = [(s / len(q), q[1:]) for s, q in zip(scores.tolist(), seqs)]
    return max(completed, key=lambda x: x[0])[1]


def dynamic_threshold(step):
    return max(0.4, 0.8 - 0.005 * step)


@torch.no_grad()
def early_exit_generate(model, image, mask, sos, eos, threshold='dynamic', use_early_exit=True,
                        max_len=250, repetition_penalty=1.0):
    """Greedy generation with early exit. After each of the first 3 layers the entropy is computed;
    if it is below the threshold, the token is emitted immediately and the skipped upper layers receive a COPY
    of the exit layer's state (so the next step and the attention do not use a stale state).
    Returns (tokens, number of layers used per token)."""
    model.eval()
    dec = model.decoder
    L = dec.num_layers
    features, hw = model.encoder(image)
    key, value = dec.attention.precompute(features)
    hidden, coverage = dec.init_state(features, mask)
    token = torch.tensor([sos], device=image.device)
    out, layers_used = [], []
    for step in range(max_len):
        thr = dynamic_threshold(step) if threshold == 'dynamic' else threshold
        context, alpha = dec.attention(hidden[-1], key, value, coverage, hw, mask)
        coverage = coverage + alpha
        x = torch.cat([dec.embedding(token), context], dim=1)
        exit_layer, logits = L - 1, None
        for i, cell in enumerate(dec.gru_layers):
            hidden[i] = cell(x, hidden[i])
            x = hidden[i]
            if use_early_exit and i < L - 1:
                lg = dec.classifier(x)
                p = F.softmax(lg.float(), dim=-1)
                if -(p * torch.log(p + 1e-9)).sum().item() < thr:
                    exit_layer, logits = i, lg
                    break
        if logits is None:
            logits = dec.classifier(hidden[-1])
        for j in range(exit_layer + 1, L):
            hidden[j] = hidden[exit_layer]
        if repetition_penalty > 1.0 and out:
            logits = logits.clone()
            idx = torch.tensor(sorted(set(out)), device=logits.device)
            v = logits[0, idx]
            logits[0, idx] = torch.where(v > 0, v / repetition_penalty, v * repetition_penalty)  # sign-aware repetition penalty
        layers_used.append(exit_layer + 1)
        tok = logits.argmax(-1).item()
        if tok == eos:
            break
        out.append(tok)
        token = torch.tensor([tok], device=image.device)
    return out, layers_used


@torch.no_grad()
def score_sequence(model, image, mask, sos, eos, tokens):
    """Forced pass over given tokens (e.g. a beam search result): for every token returns its probability,
    entropy, the layer at which early exit would emit it, and the attention map. Used for display in the app."""
    model.eval()
    dec = model.decoder
    L = dec.num_layers
    features, hw = model.encoder(image)
    key, value = dec.attention.precompute(features)
    hidden, coverage = dec.init_state(features, mask)
    seq = [sos] + list(tokens) + [eos]
    probs, entropies, exit_layers, alphas = [], [], [], []
    for t in range(len(seq) - 1):
        token = torch.tensor([seq[t]], device=image.device)
        context, alpha = dec.attention(hidden[-1], key, value, coverage, hw, mask)
        coverage = coverage + alpha
        x = torch.cat([dec.embedding(token), context], dim=1)
        exit_at = L
        for i, cell in enumerate(dec.gru_layers):
            hidden[i] = cell(x, hidden[i])
            x = hidden[i]
            if i < L - 1 and exit_at == L:
                p_i = F.softmax(dec.classifier(x).float(), -1)[0]
                if -(p_i * torch.log(p_i + 1e-9)).sum().item() < dynamic_threshold(t):
                    exit_at = i + 1
        p = F.softmax(dec.classifier(hidden[-1]).float(), -1)[0]
        probs.append(float(p[seq[t + 1]]))
        entropies.append(float(-(p * torch.log(p + 1e-9)).sum()))
        exit_layers.append(exit_at)
        alphas.append(alpha[0].float().cpu().numpy())
    return dict(probs=probs, entropies=entropies, exit_layers=exit_layers, alphas=alphas, hw=hw)


# =========================================================================================
#  Loading / saving for the app
# =========================================================================================
def save_for_inference(model, tokenizer, out_dir, extra_info=None):
    """Saves everything the app needs: weights, configuration, tokenizer and (optionally) results."""
    import json, os
    os.makedirs(out_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(out_dir, 'model.pt'))
    tokenizer.tok.save(os.path.join(out_dir, 'tokenizer.json'))
    info = {'config': model.cfg, 'vocab_size': tokenizer.vocab_size}
    info.update(extra_info or {})
    with open(os.path.join(out_dir, 'model_info.json'), 'w', encoding='utf-8') as f:
        json.dump(info, f, ensure_ascii=False, indent=1)


def load_for_inference(model_dir, device='cpu'):
    import json, os
    with open(os.path.join(model_dir, 'model_info.json'), encoding='utf-8') as f:
        info = json.load(f)
    tokenizer = TokenizerWrapper.from_file(os.path.join(model_dir, 'tokenizer.json'))
    model = Im2LatexModel(info['config'], tokenizer.vocab_size, pretrained=False)
    state = torch.load(os.path.join(model_dir, 'model.pt'), map_location=device)
    model.load_state_dict(state)
    model.to(device).eval()
    return model, tokenizer, info


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

Font rendering and augmentation code (`merlin_data.py`, used by the `'robust'` experiment).

In [ ]:
%%writefile /content/merlin_data.py
# -*- coding: utf-8 -*-
"""
merlin_data.py – data for the robust model: the same formulas rendered in many fonts, plus image augmentation.

Used only for training (Colab notebook). The web app does not need this file.

1) render_latex_batch  – renders training formulas with pdflatex in other LaTeX fonts
                          (Times, Palatino, Helvetica-math, Euler, Fourier, …) and in a bold variant
2) render_mathtext      – the same idea through matplotlib (DejaVu Sans/Serif, STIX, STIX Sans)
3) Augmenter            – random margins, stroke width, blur, rotation, JPEG, noise, contrast
"""
import io
import math
import os
import random
import re
import shutil
import subprocess
import tempfile

import numpy as np
from PIL import Image, ImageFilter, ImageOps

# ------------------------------------------------------------------------------------------
#  LaTeX fonts: name -> (preamble, bold)
# ------------------------------------------------------------------------------------------
FONT_PRESETS = {
    'cm':           ('', False),                                                   # Computer Modern
    'cm_bold':      ('', True),
    'times':        ('\\usepackage{mathptmx}', False),                             # Times
    'times_bold':   ('\\usepackage{mathptmx}', True),
    'palatino':     ('\\usepackage{mathpazo}', False),                             # Palatino
    'helvetica':    ('\\usepackage[helvet]{sfmath}\\renewcommand{\\familydefault}{\\sfdefault}', False),
    'helvetica_bold': ('\\usepackage[helvet]{sfmath}\\renewcommand{\\familydefault}{\\sfdefault}', True),
    'lmodern':      ('\\usepackage{lmodern}', False),                              # Latin Modern
    'euler':        ('\\usepackage{eulervm}', False),                              # Euler (handwriting-like)
    'fourier':      ('\\usepackage{fourier}', False),                              # Utopia
    'cmbright':     ('\\usepackage{cmbright}', False),                             # sans serif
    'kpfonts':      ('\\usepackage{kpfonts}', False),
    'newtx':        ('\\usepackage{newtxtext,newtxmath}', False),                  # Times (modern)
    'newpx':        ('\\usepackage{newpxtext,newpxmath}', False),                  # Palatino (modern)
    'arev':         ('\\usepackage{arev}', False),                                 # Bitstream Vera, sans serif
    'concrete':     ('\\usepackage{concmath}', False),
}
MATHTEXT_FONTS = ['dejavusans', 'dejavuserif', 'stix', 'stixsans']

DOC_HEAD = r"""\documentclass[12pt]{article}
\usepackage{amsmath,amssymb}
%s
\usepackage[active,tightpage]{preview}
\setlength\PreviewBorder{2pt}
\pagestyle{empty}
\begin{document}
"""


def _run(cmd, cwd, timeout):
    return subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout)


def available_fonts(candidates=None):
    """Returns the fonts whose packages exist and compile a test formula (depends on the installed TeX)."""
    ok = []
    for name in (candidates or FONT_PRESETS):
        pre, bold = FONT_PRESETS[name]
        with tempfile.TemporaryDirectory() as d:
            body = ('\\begin{preview}' + ('\\boldmath' if bold else '') +
                    '$\\displaystyle \\frac{\\alpha^2}{\\sqrt{x_1}} + \\sum_{i=1}^n \\mathcal{L}$\\end{preview}\n')
            open(os.path.join(d, 'f.tex'), 'w').write(DOC_HEAD % pre + body + '\\end{document}\n')
            try:
                r = _run(['pdflatex', '-interaction=nonstopmode', '-halt-on-error', 'f.tex'], d, 60)
                if r.returncode == 0 and os.path.exists(os.path.join(d, 'f.pdf')):
                    ok.append(name)
            except Exception:
                pass
    return ok


def crop_to_ink(img, thr=200, pad=2):
    g = np.asarray(img.convert('L'))
    ink = g < thr
    if not ink.any():
        return None
    r, c = np.where(ink.any(1))[0], np.where(ink.any(0))[0]
    g = g[max(0, r[0] - pad):r[-1] + 1 + pad, max(0, c[0] - pad):c[-1] + 1 + pad]
    return Image.fromarray(g)


def render_latex_batch(formulas, font, dpi=150, timeout=300):
    """Renders a list of formulas into one PDF (one formula = one page) and returns a list of
    PIL images (None for formulas that failed). Formulas with a LaTeX error are detected from the log
    (each formula is on its own line) and dropped, so a label and its image never get out of sync."""
    pre, bold = FONT_PRESETS[font]
    head = DOC_HEAD % pre
    first_line = head.count('\n') + 1
    lines = ['\\begin{preview}' + ('\\boldmath' if bold else '') + '$\\displaystyle ' + f + '$\\end{preview}'
             for f in formulas]
    out = [None] * len(formulas)
    with tempfile.TemporaryDirectory() as d:
        open(os.path.join(d, 'f.tex'), 'w', encoding='utf-8').write(head + '\n'.join(lines) + '\n\\end{document}\n')
        try:
            _run(['pdflatex', '-interaction=batchmode', 'f.tex'], d, timeout)
        except subprocess.TimeoutExpired:
            return out
        pdf, log = os.path.join(d, 'f.pdf'), os.path.join(d, 'f.log')
        if not os.path.exists(pdf):
            return out
        bad = set()
        txt = open(log, encoding='latin-1').read() if os.path.exists(log) else ''
        for m in re.finditer(r'^! .*?^l\.(\d+)', txt, flags=re.M | re.S):
            idx = int(m.group(1)) - first_line
            for j in (idx - 1, idx):                    # to be safe: also the previous formula (unclosed brace)
                if 0 <= j < len(formulas):
                    bad.add(j)
        info = _run(['pdfinfo', pdf], d, 60).stdout.decode('latin-1')
        pages = int(re.search(r'Pages:\s+(\d+)', info).group(1)) if 'Pages:' in info else 0
        if pages != len(formulas):                       # some formula produced no page -> do not take the risk
            if len(formulas) == 1:
                return out
            h = len(formulas) // 2                       # split in half until the problem is isolated
            return render_latex_batch(formulas[:h], font, dpi, timeout) + \
                render_latex_batch(formulas[h:], font, dpi, timeout)
        _run(['pdftoppm', '-r', str(dpi), '-gray', '-png', 'f.pdf', 'p'], d, timeout)
        files = sorted((f for f in os.listdir(d) if f.startswith('p-') and f.endswith('.png')),
                       key=lambda s: int(re.search(r'(\d+)', s).group(1)))
        for i, fn in enumerate(files[:len(formulas)]):
            if i in bad:
                continue
            with Image.open(os.path.join(d, fn)) as im:
                out[i] = crop_to_ink(im.convert('L'))
    return out


def render_mathtext(formula, fontset='dejavusans', dpi=150):
    """Rendering through matplotlib mathtext (no LaTeX). Formulas with commands mathtext does not know
    raise an error -> None."""
    import matplotlib
    matplotlib.use('Agg')
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    # mathtext misreads spaces around ^, _ and { (e.g. "q ^ { 2 }"), so the source is compacted first
    f = re.sub(r'\s*([\^_])\s*', r'\1', formula)
    f = re.sub(r'\s*([{}])\s*', r'\1', f)
    try:
        fig = Figure(figsize=(0.01, 0.01))
        FigureCanvasAgg(fig)
        fig.text(0, 0, '$' + f + '$', fontsize=14, math_fontfamily=fontset)
        buf = io.BytesIO()
        fig.savefig(buf, dpi=dpi, format='png', bbox_inches='tight', pad_inches=0.05, facecolor='white')
        buf.seek(0)
        return crop_to_ink(Image.open(buf).convert('L'))
    except Exception:
        return None


# ------------------------------------------------------------------------------------------
#  Augmentation
# ------------------------------------------------------------------------------------------
class Augmenter:
    """Random changes that resemble real screenshots: size and margins, stroke width,
    blur, slight rotation, low resolution, JPEG, noise and contrast. Works on a greyscale PIL image."""
    def __init__(self, p=0.7, seed=None, margin_always=False, rng=None):
        # rng=random (the global module) in a DataLoader: PyTorch seeds it separately in every worker
        self.p, self.margin_always = p, margin_always
        self.rng = rng if rng is not None else random.Random(seed)

    def margins(self, img):
        r = self.rng
        w, h = img.size
        frac = r.uniform(0.3, 0.95)                      # share of the height taken by the formula
        v = int(round(h * (1 / frac - 1) / 2))
        left, right = r.randint(2, max(3, h // 3)), r.randint(2, max(3, h))
        top = int(round(v * r.uniform(0.6, 1.4))); bottom = max(0, 2 * v - top)
        return ImageOps.expand(img, (left, top, right, bottom), fill=255)

    def __call__(self, img):
        r = self.rng
        img = img.convert('L')
        if self.margin_always or r.random() < 0.5:
            img = self.margins(img)
        if r.random() > self.p:
            return img
        base = img
        if r.random() < 0.3 and min(img.size) > 12:
            img = img.filter(ImageFilter.MinFilter(3))      # thicker strokes (dark ink spreads)
        if r.random() < 0.3:
            img = img.rotate(r.uniform(-2.0, 2.0), resample=Image.BILINEAR, expand=True, fillcolor=255)
        if r.random() < 0.3:
            s = r.uniform(0.35, 0.8)                        # down to low resolution and back
            w, h = img.size
            img = img.resize((max(8, int(w * s)), max(8, int(h * s))), Image.BILINEAR).resize((w, h), Image.BILINEAR)
        if r.random() < 0.3:
            img = img.filter(ImageFilter.GaussianBlur(r.uniform(0.3, 1.1)))
        if r.random() < 0.3:                                # grey background / faint ink
            a = np.asarray(img, dtype=np.float32) / 255.0
            bg, fg = r.uniform(190, 255), r.uniform(0, 90)
            img = Image.fromarray((fg + a * (bg - fg)).clip(0, 255).astype(np.uint8))
        if r.random() < 0.2:
            a = np.asarray(img, dtype=np.float32)
            a = a + np.random.default_rng(r.randrange(1 << 30)).normal(0, r.uniform(3, 12), a.shape)
            img = Image.fromarray(a.clip(0, 255).astype(np.uint8))
        if r.random() < 0.3:
            buf = io.BytesIO()
            img.save(buf, format='JPEG', quality=r.randint(25, 85))
            buf.seek(0)
            img = Image.open(buf).convert('L')
        # safeguard: if the formula has almost vanished (too blurred/faded), return the undistorted image
        ink0 = (np.asarray(base) < 128).sum()
        if ink0 > 0 and (np.asarray(img) < 128).sum() < 0.3 * ink0:
            return base
        return img


# ------------------------------------------------------------------------------------------
#  Workers for parallel rendering (ProcessPoolExecutor)
# ------------------------------------------------------------------------------------------
def latex_job(args):
    """(font, dpi, [(id, formula)], out_dir, prefix) -> list of ids that were saved."""
    font, dpi, items, out_dir, prefix = args
    ims = render_latex_batch([f for _, f in items], font, dpi=dpi)
    done = []
    for (i, _), im in zip(items, ims):
        if im is not None and im.width > 4 and im.height > 4:
            im.save(os.path.join(out_dir, f'{prefix}{i}.png'))
            done.append(i)
    return done


def mathtext_job(args):
    """(fontset, dpi, [(id, formula)], out_dir, prefix) -> list of ids that were saved."""
    fontset, dpi, items, out_dir, prefix = args
    done = []
    for i, f in items:
        im = render_mathtext(f, fontset, dpi=dpi)
        if im is not None and im.width > 4 and im.height > 4:
            im.save(os.path.join(out_dir, f'{prefix}{i}.png'))
            done.append(i)
    return done

In [ ]:
# CELL 2b: Quick check of the model code (random weights, ~30 s) – before the long training run
import sys, importlib, torch
sys.path.insert(0, '/content')
import merlin_model; importlib.reload(merlin_model)
from merlin_model import *
cfg = dict(DEFAULT_CONFIG, **MODEL_CFG)
m = Im2LatexModel(cfg, 50, pretrained=False)
imgs = [torch.rand(1, 96, w) for w in (120, 300, 768)] if cfg['preprocess'] == 'aspect' else [torch.rand(1, 224, 224)] * 3
images, mask = collate_images(imgs, cfg)
caps = torch.randint(4, 50, (3, 12)); caps[:, 0] = 1
out = m(images, mask, caps[:, :-1], ss_prob=0.2)
loss, _ = total_loss(m, out, caps[:, 1:], 0, KDLoss(pad_idx=0)); loss.backward()
one, one_mask = collate_images(imgs[:1], cfg)
print("training:", round(loss.item(), 3),
      "| greedy:", len(greedy_batch(m, images, mask, 1, 2, 8)),
      "| beam:", len(beam_search(m, one, one_mask, 1, 2, 3, 8)),
      "| early exit:", len(early_exit_generate(m, one, one_mask, 1, 2, max_len=8)[0]),
      "| map:", m.encoder(one)[1])
del m
import merlin_data; importlib.reload(merlin_data)
merlin_data.Augmenter(p=1.0, seed=0, margin_always=True)(Image.new('L', (60, 20), 255))
print("Model code works.")

## 3. Data and tokenizer

In [ ]:
# CELL 3: Data (im2latex-100k) and a tokenizer trained ONLY on training formulas
import sys, tarfile
import pandas as pd
import torch
sys.path.insert(0, '/content')
import importlib, merlin_model
importlib.reload(merlin_model)
from merlin_model import *

DATA_DIR = '/content/latex_data'
if not os.path.exists(DATA_DIR):
    # Kaggle key: 1) kaggle.json in MERLIN on Google Drive, or 2) Colab Secrets KAGGLE_USERNAME and KAGGLE_KEY
    kj = os.path.expanduser('~/.kaggle/kaggle.json')
    if not os.path.exists(kj) and not os.environ.get('KAGGLE_KEY'):
        os.makedirs(os.path.dirname(kj), exist_ok=True)
        if os.path.exists(os.path.join(SAVE_ROOT, 'kaggle.json')):
            os.system(f"cp '{os.path.join(SAVE_ROOT, 'kaggle.json')}' {kj} && chmod 600 {kj}")
        else:
            try:
                from google.colab import userdata
                os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
                os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
            except Exception:
                print("No Kaggle key found – if the download fails, see the README section on the Kaggle key.")
    !kaggle datasets download -d shahrukhkhan/im2latex100k -p /content
    !unzip -q /content/im2latex100k.zip -d /content/latex_data
tar_path = os.path.join(DATA_DIR, 'formula_images_processed.tar.gz')
if os.path.exists(tar_path) and not os.path.exists(os.path.join(DATA_DIR, 'formula_images_processed')):
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(path=DATA_DIR)
img_dir = next(root for root, _, files in os.walk(DATA_DIR) if any(f.endswith('.png') for f in files))

def load_split(csv_path):
    df = pd.read_csv(csv_path)
    cols = [str(c).lower() for c in df.columns]
    img_col = df.columns[cols.index('image')] if 'image' in cols else df.columns[0]
    form_col = df.columns[cols.index('formula')] if 'formula' in cols else df.columns[1]
    rows = []
    for a, b in zip(df[img_col].astype(str), df[form_col].astype(str)):
        img, f = (b, a) if ('\\' in a or len(a) > 60) else (a, b)
        rows.append({'image': img.replace('.png', '') + '.png', 'formula': ' '.join(f.split())})
    return pd.DataFrame(rows)

train_df = load_split(os.path.join(DATA_DIR, 'im2latex_train.csv'))
val_df = load_split(os.path.join(DATA_DIR, 'im2latex_validate.csv'))
test_df = load_split(os.path.join(DATA_DIR, 'im2latex_test.csv'))
print(f"Train {len(train_df)} | validation {len(val_df)} | test {len(test_df)}")

TOK_PATH = os.path.join(SAVE_ROOT, 'tokenizer.json')        # the same tokenizer for all experiments
if not os.path.exists(TOK_PATH):
    with open('/content/train_formulas.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(train_df['formula']))
    build_tokenizer('/content/train_formulas.txt').save(TOK_PATH)
tokenizer = TokenizerWrapper.from_file(TOK_PATH)
rt = np.mean([tokenizer.decode(tokenizer.encode(f)) == f for f in train_df['formula'][:2000]])
save_results({'tokenizer': {'vocab_size': tokenizer.vocab_size, 'roundtrip_train': float(rt)}})
print(f"Vocabulary: {tokenizer.vocab_size} tokens | exact text round-trip: {100*rt:.2f} %")

### 3b. Formulas in many fonts and the robustness test (training fonts only for `EXPERIMENT = 'robust'`)

In [ ]:
# CELL 3b: Robustness test set (for ALL experiments) + the same training formulas in many fonts ('robust' only)
# The robustness test is built once (≈ 10–15 min, on the first run of any experiment), the training fonts once
# for 'robust' (≈ 25–35 min). Everything is stored on Drive; later runs only unpack the archives (≈ 1–2 min).
import tarfile
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor

SYNTH_DIR, FONTTEST_DIR = '/content/synth', '/content/font_test'
SYNTH_TAR, FONTTEST_TAR = os.path.join(SAVE_ROOT, 'synth_fonts.tar'), os.path.join(SAVE_ROOT, 'font_test.tar')
SYNTH_META, FONTTEST_META = os.path.join(SAVE_ROOT, 'synth_meta.json'), os.path.join(SAVE_ROOT, 'font_test_meta.json')
synth_index = {}

def untar(tar_path, dest):
    if os.path.exists(tar_path) and not os.path.exists(dest):
        with tarfile.open(tar_path) as t:
            t.extractall('/content')

FONTS_JSON = os.path.join(SAVE_ROOT, 'fonts.json')       # the same font split for all experiments
need_synth = bool(EXP.get('synth')) and not os.path.exists(SYNTH_META)
need_test = not os.path.exists(FONTTEST_META)                # the robustness test is built by the FIRST run of any experiment

if need_synth or need_test:
    sys.path.insert(0, '/content')
    import merlin_data; importlib.reload(merlin_data)
    from merlin_data import (FONT_PRESETS, MATHTEXT_FONTS, Augmenter, available_fonts, latex_job, mathtext_job)
    if not os.path.exists('/usr/share/texlive/texmf-dist/tex/latex/fourier'):
        print("Installing extra LaTeX fonts (texlive-fonts-extra, ≈ 5 min)...")
        os.system("apt-get install -y -qq texlive-fonts-extra > /dev/null 2>&1")
    if os.path.exists(FONTS_JSON):
        with open(FONTS_JSON) as f:
            fj = json.load(f)
        train_latex, train_mt, HELDOUT_LATEX, HELDOUT_MT = fj['train_latex'], fj['train_mt'], fj['heldout_latex'], fj['heldout_mt']
    else:
        fonts = available_fonts()
        HELDOUT_LATEX = next((f for f in ('fourier', 'euler', 'kpfonts', 'palatino') if f in fonts), None)
        HELDOUT_MT = 'stixsans'                              # and one matplotlib font for testing only
        train_latex = [f for f in fonts if f != HELDOUT_LATEX]
        train_mt = [m for m in MATHTEXT_FONTS if m != HELDOUT_MT]
        with open(FONTS_JSON, 'w') as f:
            json.dump({'train_latex': train_latex, 'train_mt': train_mt,
                       'heldout_latex': HELDOUT_LATEX, 'heldout_mt': HELDOUT_MT}, f)
    print("Training fonts:", train_latex, train_mt, "| unseen fonts for testing:", HELDOUT_LATEX, HELDOUT_MT)
    rng = random.Random(0)

    def run_jobs(fn, jobs, desc):
        done = []
        with ProcessPoolExecutor(max(2, os.cpu_count() or 2)) as ex:
            for r in tqdm(ex.map(fn, jobs), total=len(jobs), desc=desc):
                done += r
        return set(done)

    # --- training: every training formula in one random LaTeX font + 30,000 in matplotlib fonts
    if need_synth:
        os.makedirs(SYNTH_DIR, exist_ok=True)
        forms = list(train_df['formula'])
        by_font = {}
        for i in range(len(forms)):
            by_font.setdefault(rng.choice(train_latex), []).append(i)
        jobs = []
        for font, ids in by_font.items():
            for k in range(0, len(ids), 400):
                jobs.append((font, rng.choice([110, 130, 150, 180, 220]),
                             [(i, forms[i]) for i in ids[k:k + 400]], SYNTH_DIR, 'l_'))
        ok_l = run_jobs(latex_job, jobs, "LaTeX fonts")
        mt_ids = rng.sample(range(len(forms)), min(30000, len(forms)))
        jobs = [(rng.choice(train_mt), rng.choice([110, 150, 200]), [(i, forms[i]) for i in mt_ids[k:k + 200]],
                 SYNTH_DIR, 'm_') for k in range(0, len(mt_ids), 200)]
        ok_m = run_jobs(mathtext_job, jobs, "matplotlib fonts")
        index = {str(i): [p for p, ok in ((f'l_{i}.png', i in ok_l), (f'm_{i}.png', i in ok_m)) if ok]
                 for i in range(len(forms))}
        with tarfile.open(SYNTH_TAR, 'w') as t:
            t.add(SYNTH_DIR, arcname='synth')
        with open(SYNTH_META, 'w') as f:
            json.dump({'index': index, 'train_latex': train_latex, 'train_mt': train_mt,
                       'n_latex': len(ok_l), 'n_mathtext': len(ok_m)}, f)
        print(f"Rendered: {len(ok_l)} formulas in LaTeX fonts, {len(ok_m)} in matplotlib fonts")

    # --- robustness test: test formulas (never seen by the model) in training fonts
    #     and in fonts the model has NOT seen, with distortions (fixed seed -> the same test for every model)
    if need_test:
        os.makedirs(FONTTEST_DIR, exist_ok=True)
        tforms = list(test_df['formula'])
        trng = random.Random(7)
        pick = trng.sample(range(len(tforms)), 1500)
        known, unseen = pick[:1000], pick[1000:]
        plan = [(i, 'known_fonts', trng.choice(train_latex) if trng.random() < 0.7 else 'mt:' + trng.choice(train_mt))
                for i in known]
        plan += [(i, 'unseen_fonts', (HELDOUT_LATEX if HELDOUT_LATEX and trng.random() < 0.6 else 'mt:' + HELDOUT_MT))
                 for i in unseen]
        raw = {}
        for split_font in sorted({p[2] for p in plan}):
            ids = [i for i, _, f in plan if f == split_font]
            if split_font.startswith('mt:'):
                jobs = [(split_font[3:], 150, [(i, tforms[i]) for i in ids[k:k + 200]], FONTTEST_DIR, 'raw_')
                        for k in range(0, len(ids), 200)]
                ok = run_jobs(mathtext_job, jobs, split_font)
            else:
                jobs = [(split_font, 150, [(i, tforms[i]) for i in ids[k:k + 200]], FONTTEST_DIR, 'raw_')
                        for k in range(0, len(ids), 200)]
                ok = run_jobs(latex_job, jobs, split_font)
            raw.update({i: split_font for i in ok})
        aug_test = Augmenter(p=0.8, seed=123, margin_always=True)
        meta = {'known_fonts': [], 'unseen_fonts': []}
        for i, split, font in plan:
            if i not in raw:
                continue
            src = os.path.join(FONTTEST_DIR, f'raw_{i}.png')
            im = aug_test(Image.open(src).convert('L'))
            im.save(os.path.join(FONTTEST_DIR, f't_{i}.png')); os.remove(src)
            meta[split].append({'id': i, 'file': f't_{i}.png', 'font': font})
        with tarfile.open(FONTTEST_TAR, 'w') as t:
            t.add(FONTTEST_DIR, arcname='font_test')
        with open(FONTTEST_META, 'w') as f:
            json.dump(meta, f)
        print("Robustness test:", {k: len(v) for k, v in meta.items()})

untar(FONTTEST_TAR, FONTTEST_DIR)
if EXP.get('synth'):
    untar(SYNTH_TAR, SYNTH_DIR)
    with open(SYNTH_META) as f:
        sm = json.load(f)
    synth_index = {int(k): v for k, v in sm['index'].items()}
    save_results({'synth': {k: sm[k] for k in ('train_latex', 'train_mt', 'n_latex', 'n_mathtext')}})
    print(f"Synthetic images ready: {sm['n_latex']} LaTeX + {sm['n_mathtext']} matplotlib")
else:
    print("This experiment trains on the original images only (extra training fonts are used by 'robust'); "
          "the robustness test is ready and is measured in cell 7b.")

## 4. Dataset

In [ ]:
# CELL 4: Dataset and DataLoader (preprocessing depends on the experiment; 'robust' adds fonts and augmentation)
from torch.utils.data import Dataset, DataLoader

preprocessor = ImagePreprocessor(MODEL_CFG)
if EXP.get('aug'):
    from merlin_data import Augmenter
    AUG_ORIG = Augmenter(p=0.7, rng=random)                       # original dataset images
    AUG_SYN = Augmenter(p=0.7, rng=random, margin_always=True)    # synthetic images are cropped to the edge -> always add margins

class Im2LatexDataset(Dataset):
    """train=True and EXPERIMENT='robust': for every formula pick at random the original image (45 %),
    the same formula in another LaTeX font (40 %) or in a matplotlib font (15 %), then apply augmentation."""
    def __init__(self, df, train=False):
        self.df, self.train = df.reset_index(drop=True), train
    def __len__(self):
        return len(self.df)
    def load(self, i):
        orig = os.path.join(img_dir, self.df['image'][i])
        if not (self.train and EXP.get('synth')):
            return Image.open(orig).convert('L'), False
        opts, w = [(orig, False)], [0.45]
        for f in synth_index.get(i, []):
            opts.append((os.path.join(SYNTH_DIR, f), True)); w.append(0.40 if f.startswith('l_') else 0.15)
        path, syn = random.choices(opts, weights=w)[0]
        return Image.open(path).convert('L'), syn
    def __getitem__(self, i):
        img, syn = self.load(i)
        if self.train and EXP.get('aug'):
            img = (AUG_SYN if syn else AUG_ORIG)(img)
        return preprocessor(img), torch.tensor(tokenizer.encode(self.df['formula'][i]), dtype=torch.long), i

def collate(batch):
    imgs, toks, ids = zip(*batch)
    images, mask = collate_images(imgs, MODEL_CFG)
    tokens = torch.nn.utils.rnn.pad_sequence(toks, batch_first=True, padding_value=tokenizer.pad)
    return images, mask, tokens, torch.tensor(ids)

use_cuda = torch.cuda.is_available()
train_ds, val_ds, test_ds = Im2LatexDataset(train_df, train=True), Im2LatexDataset(val_df), Im2LatexDataset(test_df)
train_loader = DataLoader(train_ds, batch_size=TRAIN['batch_size'], shuffle=True, collate_fn=collate,
                          num_workers=2, pin_memory=use_cuda, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=TRAIN['batch_size'], shuffle=False, collate_fn=collate,
                        num_workers=2, pin_memory=use_cuda)
test_loader = DataLoader(test_ds, batch_size=TRAIN['batch_size'], shuffle=False, collate_fn=collate,
                         num_workers=2, pin_memory=use_cuda)
print(f"Batches per epoch: {len(train_loader)}" + (" | fonts + augmentation: ON" if EXP.get('synth') else ""))

## 5. Training

In [ ]:
# CELL 5: Training (KD without <PAD>, scheduled sampling, resume after interruption)
import torch.optim as optim
from tqdm.auto import tqdm
from torch.amp import autocast, GradScaler

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Im2LatexModel(MODEL_CFG, tokenizer.vocab_size).to(DEVICE)
kd_loss = KDLoss(TRAIN['kd_T'], TRAIN['kd_alpha'], tokenizer.pad)
optimizer = optim.AdamW(model.parameters(), lr=TRAIN['lr'], weight_decay=TRAIN['weight_decay'])
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=TRAIN['max_lr'],
                                                steps_per_epoch=len(train_loader), epochs=TRAIN['epochs'])
scaler = GradScaler('cuda')
CKPT_LAST, CKPT_BEST = os.path.join(RUN_DIR, 'checkpoint_last.pt'), os.path.join(RUN_DIR, 'model_best.pt')

def ss_prob(epoch):
    """Share of the model's own predictions used as input: 0 until ss_start_epoch, then linearly up to ss_max."""
    e0, E = TRAIN['ss_start_epoch'] - 1, TRAIN['epochs'] - 1
    if TRAIN['ss_max'] <= 0 or epoch < e0:
        return 0.0
    return TRAIN['ss_max'] * min(1.0, (epoch - e0 + 1) / max(1, E - e0 + 1))

history = {'train_loss': [], 'train_ce_top': [], 'val_ce': [], 'val_acc_layers': [], 'ss_prob': [],
           'epoch_minutes': [], 'best_epoch': None}
start_epoch, best_val = 0, float('inf')
if os.path.exists(CKPT_LAST):
    ck = torch.load(CKPT_LAST, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
    scheduler.load_state_dict(ck['scheduler']); scaler.load_state_dict(ck['scaler'])
    history, start_epoch, best_val = ck['history'], ck['epoch'] + 1, ck['best_val']
    print(f"Resuming from epoch {start_epoch + 1}")

@torch.no_grad()
def validate():
    model.eval()
    L = model.decoder.num_layers
    ce, correct, n = 0.0, [0] * L, 0
    for images, mask, tokens, _ in val_loader:
        images, mask, tokens = images.to(DEVICE), mask.to(DEVICE), tokens.to(DEVICE)
        with autocast('cuda'):
            out = model(images, mask, tokens[:, :-1])
        y = tokens[:, 1:].reshape(-1); valid = y != tokenizer.pad; n += int(valid.sum())
        for i in range(L):
            lg = out[i].reshape(-1, tokenizer.vocab_size).float()[valid]
            correct[i] += (lg.argmax(-1) == y[valid]).sum().item()
            if i == L - 1:
                ce += F.cross_entropy(lg, y[valid], reduction='sum').item()
    return ce / n, [100.0 * c / n for c in correct]

for epoch in range(start_epoch, TRAIN['epochs']):
    model.train()
    p_ss, t0, run_loss, run_ce = ss_prob(epoch), time.time(), 0.0, 0.0
    bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{TRAIN['epochs']} (ss={p_ss:.2f})")
    for images, mask, tokens, _ in bar:
        images, mask, tokens = images.to(DEVICE), mask.to(DEVICE), tokens.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            out = model(images, mask, tokens[:, :-1], ss_prob=p_ss)
            loss, ce_top = total_loss(model, out, tokens[:, 1:], tokenizer.pad, kd_loss, TRAIN['student_weight'])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        run_loss += loss.item(); run_ce += ce_top.item()
        bar.set_postfix(loss=f"{loss.item():.3f}")
    val_ce, val_acc = validate()
    for k, v in (('train_loss', run_loss / len(train_loader)), ('train_ce_top', run_ce / len(train_loader)),
                 ('val_ce', val_ce), ('val_acc_layers', val_acc), ('ss_prob', p_ss),
                 ('epoch_minutes', (time.time() - t0) / 60)):
        history[k].append(v)
    print(f"Epoch {epoch+1}: train {history['train_loss'][-1]:.4f} | val CE {val_ce:.4f} | "
          f"token accuracy per layer {[round(a, 1) for a in val_acc]}")
    if val_ce < best_val:
        best_val = val_ce; history['best_epoch'] = epoch + 1
        torch.save(model.state_dict(), CKPT_BEST)
        print("  --> new best model")
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                'scaler': scaler.state_dict(), 'history': history, 'epoch': epoch, 'best_val': best_val}, CKPT_LAST)
    save_results({'history': history})

## 6. Evaluation tools

In [ ]:
# Evaluation tools (rendering, image comparison, metrics, statistics)
import os, re, math, shutil, subprocess, tempfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from PIL import Image
import Levenshtein

TEX_TEMPLATE = r"""\documentclass[preview,border=4pt]{standalone}
\usepackage{amsmath,amssymb}
\begin{document}
$\displaystyle %s$
\end{document}
"""

def render_formula(formula, out_png, dpi=150, timeout=10):
    """Compiles a formula with pdflatex and converts the PDF to PNG. Returns True on success."""
    if not formula.strip():
        return False
    with tempfile.TemporaryDirectory() as d:
        with open(os.path.join(d, 'f.tex'), 'w', encoding='utf-8') as fh:
            fh.write(TEX_TEMPLATE % formula)
        try:
            r = subprocess.run(['pdflatex', '-interaction=nonstopmode', '-halt-on-error', 'f.tex'],
                               cwd=d, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=timeout)
            if r.returncode != 0 or not os.path.exists(os.path.join(d, 'f.pdf')):
                return False
            subprocess.run(['pdftoppm', '-r', str(dpi), '-gray', '-singlefile', '-png', 'f.pdf', 'img'],
                           cwd=d, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=timeout)
            src = os.path.join(d, 'img.png')
            if not os.path.exists(src):
                return False
            shutil.move(src, out_png)
            return True
        except Exception:
            return False


def _load_binary(png):
    a = np.array(Image.open(png).convert('L')) < 128          # True = black pixel (ink)
    if not a.any():
        return a[:0, :0]
    rows, cols = np.where(a.any(1))[0], np.where(a.any(0))[0]
    return a[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]


def _dilate(a):
    """Binary dilation by 1 pixel (3x3), without extra libraries."""
    p = np.pad(a, 1)
    out = np.zeros_like(a)
    for dy in (0, 1, 2):
        for dx in (0, 1, 2):
            out |= p[dy:dy + a.shape[0], dx:dx + a.shape[1]]
    return out


def images_match(png_a, png_b, size_tol=2, max_mismatch=0.01):
    """Image match (after Deng et al. 2017, the '-ws' variant):
    both images are binarised and cropped, empty columns (spaces) are removed, and they are compared
    pixel by pixel with a 1-pixel tolerance. A formula is correct if fewer than 1 % of the ink
    pixels differ."""
    a, b = _load_binary(png_a), _load_binary(png_b)
    if a.size == 0 or b.size == 0:
        return False
    a, b = a[:, a.any(0)], b[:, b.any(0)]                       # remove empty columns
    if abs(a.shape[0] - b.shape[0]) > size_tol or abs(a.shape[1] - b.shape[1]) > size_tol:
        return False
    H, W = max(a.shape[0], b.shape[0]), max(a.shape[1], b.shape[1])
    A = np.zeros((H, W), bool); A[:a.shape[0], :a.shape[1]] = a
    B = np.zeros((H, W), bool); B[:b.shape[0], :b.shape[1]] = b
    miss = (A & ~_dilate(B)).sum() + (B & ~_dilate(A)).sum()
    return miss <= max_mismatch * max(1, A.sum() + B.sum())


# ----------------------------- text metrics -----------------------------
def nospace(s):
    return s.replace(' ', '')

def token_edit_distance(gold_tokens, pred_tokens):
    """Levenshtein distance over LaTeX tokens (every token is mapped to one character)."""
    vocab = {}
    enc = lambda toks: ''.join(chr(0xE000 + vocab.setdefault(t, len(vocab))) for t in toks)
    return Levenshtein.distance(enc(gold_tokens), enc(pred_tokens))

def corpus_bleu(refs, hyps, max_n=4):
    """Corpus BLEU-4 over tokens (Papineni et al. 2002), in percent."""
    clipped, total = [0] * max_n, [0] * max_n
    ref_len = hyp_len = 0
    for r, h in zip(refs, hyps):
        ref_len += len(r); hyp_len += len(h)
        for n in range(1, max_n + 1):
            hc = Counter(tuple(h[i:i + n]) for i in range(len(h) - n + 1))
            rc = Counter(tuple(r[i:i + n]) for i in range(len(r) - n + 1))
            clipped[n - 1] += sum(min(c, rc[g]) for g, c in hc.items())
            total[n - 1] += max(len(h) - n + 1, 0)
    if hyp_len == 0 or min(clipped) == 0:
        return 0.0
    log_p = sum(math.log(clipped[i] / total[i]) for i in range(max_n)) / max_n
    bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / hyp_len)
    return 100.0 * bp * math.exp(log_p)

def wilson_ci(k, n, z=1.96):
    """95 % confidence interval for a proportion (Wilson), in percent."""
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (100 * max(0.0, c - h), 100 * min(1.0, c + h))

def mcnemar_exact(a_correct, b_correct):
    """Exact McNemar test for paired binary outcomes. Returns (b, c, p)."""
    b = sum(1 for x, y in zip(a_correct, b_correct) if x and not y)
    c = sum(1 for x, y in zip(a_correct, b_correct) if y and not x)
    n = b + c
    if n == 0:
        return b, c, 1.0
    k = min(b, c)
    p = 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return b, c, min(1.0, p)

def mean_ci(values):
    v = np.asarray(values, float)
    if len(v) == 0:
        return (0.0, 0.0, 0.0)
    m = float(v.mean())
    h = 1.96 * float(v.std(ddof=1)) / math.sqrt(len(v)) if len(v) > 1 else 0.0
    return (m, m - h, m + h)


def text_metrics(golds, preds):
    """golds/preds: lists of strings in normalised form (tokens separated by spaces)."""
    em = [nospace(g) == nospace(p) for g, p in zip(golds, preds)]
    char_ed = [Levenshtein.distance(nospace(g), nospace(p)) for g, p in zip(golds, preds)]
    tok_ed = [token_edit_distance(g.split(), p.split()) for g, p in zip(golds, preds)]
    norm_ed = [e / max(1, len(g.split())) for e, g in zip(tok_ed, golds)]
    n = len(golds)
    return {
        'n': n,
        'em': 100.0 * sum(em) / max(1, n), 'em_ci': wilson_ci(sum(em), n),
        'bleu': corpus_bleu([g.split() for g in golds], [p.split() for p in preds]),
        'char_edit': mean_ci(char_ed), 'tok_edit': mean_ci(tok_ed),
        'norm_tok_edit': mean_ci(norm_ed),
        'per_sample_em': em,
    }


def render_metrics(golds, preds, ids, render_dir, tag, workers=None):
    """Compilation + image match. Ground-truth formulas are rendered once and cached."""
    os.makedirs(render_dir, exist_ok=True)
    workers = workers or max(2, os.cpu_count() or 2)

    def gold_png(i, g):
        p = os.path.join(render_dir, f'gold_{i}.png')
        ok_flag = p + '.fail'
        if os.path.exists(p):
            return p
        if os.path.exists(ok_flag):
            return None
        if render_formula(g, p):
            return p
        open(ok_flag, 'w').close()
        return None

    def one(args):
        i, g, pr = args
        gp = gold_png(i, g)
        if nospace(g) == nospace(pr):                       # identical text -> identical image
            return (gp is not None, gp is not None, gp is not None)
        pp = os.path.join(tempfile.gettempdir(), f'{tag}_{i}.png')   # temporary, local
        compiled = render_formula(pr, pp)
        match = bool(compiled and gp is not None and images_match(gp, pp))
        if os.path.exists(pp):
            os.remove(pp)
        return (compiled, match, gp is not None)

    with ThreadPoolExecutor(workers) as ex:
        res = list(ex.map(one, zip(ids, golds, preds)))
    compiled = [r[0] for r in res]
    gold_ok = [r[2] for r in res]
    match = [r[1] for r, ok in zip(res, gold_ok) if ok]          # only where the ground-truth formula renders
    n_ok = sum(gold_ok)
    return {
        'compile': 100.0 * sum(compiled) / max(1, len(compiled)),
        'compile_ci': wilson_ci(sum(compiled), len(compiled)),
        'img_match': 100.0 * sum(match) / max(1, n_ok),
        'img_match_ci': wilson_ci(sum(match), n_ok),
        'n_gold_rendered': n_ok,
        'per_sample_img': [r[1] if ok else None for r, ok in zip(res, gold_ok)],
    }

## 7. Evaluation

In [ ]:
# CELL 7: Evaluation – full test set + beam search and early exit on the same 1,000 random examples
model.load_state_dict(torch.load(CKPT_BEST, map_location=DEVICE))
model.eval()
R = load_results()
golds = list(test_df['formula'])
SOS, EOS = tokenizer.sos, tokenizer.eos
LEN_BUCKETS = [(1, 25), (26, 50), (51, 75), (76, 100), (101, 150), (151, 10**6)]

def strip(m):
    return {k: v for k, v in m.items() if not k.startswith('per_sample')}
def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
def single(k):
    img, toks, i = test_ds[k]
    images, mask, _, _ = collate([(img, toks, i)])
    return images.to(DEVICE), mask.to(DEVICE)

# --- 1) Full test set, greedy (batched): text + image match ---------------------------------
if 'full_greedy' not in R:
    preds, t0 = [None] * len(test_ds), time.time()
    for images, mask, tokens, ids in tqdm(test_loader, desc="Full test set"):
        with autocast('cuda'):
            seqs = greedy_batch(model, images.to(DEVICE), mask.to(DEVICE), SOS, EOS, MAX_GEN_LEN)
        for i, s in zip(ids.tolist(), seqs):
            preds[i] = tokenizer.decode(s)
    gen_sec = time.time() - t0
    tm = text_metrics(golds, preds)
    res = {**strip(tm), 'seconds_generation': gen_sec}
    lens = [len(g.split()) for g in golds]
    res['by_len'] = [{'lo': lo, 'hi': hi, 'n': sum(lo <= n <= hi for n in lens),
                      'em': 100.0 * np.mean([e for e, n in zip(tm['per_sample_em'], lens) if lo <= n <= hi] or [0])}
                     for lo, hi in LEN_BUCKETS]
    if EVAL['full_image_match']:
        print("Image match on the full test set (rendering ~10,000 formulas, about 45-60 min)...")
        rm = render_metrics(golds, preds, list(range(len(golds))), RENDER_DIR, 'full')
        res.update(strip(rm))
        res['by_len_img'] = [100.0 * np.mean([x for x, n in zip(rm['per_sample_img'], lens)
                                              if x is not None and lo <= n <= hi] or [0]) for lo, hi in LEN_BUCKETS]
        res['per_sample_img'] = rm['per_sample_img']
    R = save_results({'full_greedy': {k: v for k, v in res.items() if k != 'per_sample_img'},
                      'full_greedy_preds': preds, 'full_greedy_em': tm['per_sample_em'],
                      'full_greedy_img': res.get('per_sample_img')})
    print({k: v for k, v in R['full_greedy'].items() if not isinstance(v, list)})

# --- 2) The same random sample as in the thesis (seed 42): greedy, early exit, beam -----------
subset = sorted(random.Random(EVAL['subset_seed']).sample(range(len(test_ds)), EVAL['n_subset']))
golds_sub = [golds[k] for k in subset]
R = save_results({'subset': subset})

def run_method(name, fn):
    global R
    if name in R.get('methods', {}):
        return
    preds, layers, times = [], [], []
    for k in tqdm(subset, desc=name):
        sync(); t0 = time.time()
        p, ls = fn(k)
        sync(); times.append(time.time() - t0)
        preds.append(p); layers += ls or []
    tm, rm = text_metrics(golds_sub, preds), render_metrics(golds_sub, preds, subset, RENDER_DIR, name)
    res = {**strip(tm), **strip(rm), 'sec_per_formula': float(np.mean(times)),
           'em_list': tm['per_sample_em'], 'img_list': rm['per_sample_img'], 'preds': preds}
    if layers:
        res['avg_layers'] = float(np.mean(layers))
        res['layer_hist'] = [int(sum(1 for l in layers if l == j)) for j in range(1, 5)]
    M = R.get('methods', {}); M[name] = res
    R = save_results({'methods': M})
    print(name, {k: round(v, 3) for k, v in strip(res).items() if isinstance(v, float)})

run_method('greedy', lambda k: (lambda o: (tokenizer.decode(o[0]), o[1]))(
    early_exit_generate(model, *single(k), SOS, EOS, use_early_exit=False, max_len=MAX_GEN_LEN)))
run_method('early_exit', lambda k: (lambda o: (tokenizer.decode(o[0]), o[1]))(
    early_exit_generate(model, *single(k), SOS, EOS, threshold='dynamic', use_early_exit=True, max_len=MAX_GEN_LEN)))
run_method('beam', lambda k: (tokenizer.decode(beam_search(model, *single(k), SOS, EOS, EVAL['beam_width'], MAX_GEN_LEN)), None))

tests = {}
M = R['methods']
for a, b in (('greedy', 'beam'), ('greedy', 'early_exit')):
    tests[f'{a}__{b}__em'] = mcnemar_exact(M[a]['em_list'], M[b]['em_list'])
    pairs = [(x, y) for x, y in zip(M[a]['img_list'], M[b]['img_list']) if x is not None and y is not None]
    tests[f'{a}__{b}__img'] = mcnemar_exact([x for x, _ in pairs], [y for _, y in pairs])
R = save_results({'mcnemar': tests})
print("McNemar:", tests)

# --- 3) Early exit: accuracy and speed for different thresholds ---------------------------
if 'ee_curve' not in R:
    curve = []
    for thr in ['none'] + EVAL['ee_thresholds']:
        em, layers, times = [], [], []
        for k in tqdm(subset[:EVAL['ee_curve_n']], desc=f"threshold {thr}"):
            sync(); t0 = time.time()
            out, ls = early_exit_generate(model, *single(k), SOS, EOS, threshold=0 if thr == 'none' else thr,
                                          use_early_exit=thr != 'none', max_len=MAX_GEN_LEN)
            sync(); times.append(time.time() - t0)
            em.append(nospace(tokenizer.decode(out)) == nospace(golds[k])); layers += ls
        curve.append({'threshold': thr, 'em': 100.0 * np.mean(em), 'avg_layers': float(np.mean(layers)),
                      'sec_per_formula': float(np.mean(times))})
        print(curve[-1])
    R = save_results({'ee_curve': curve})
print("Evaluation finished:", RESULTS_PATH)

### 7b. Robustness test: other fonts and distortions

In [ ]:
# CELL 7b: Robustness test – test formulas in other fonts and with distortions (screenshot, JPEG, blur…)
# Measures this model (and the baseline, if this is not the baseline) on the SAME images from cell 3b. Results are cached.
if os.path.exists(FONTTEST_META):
    untar(FONTTEST_TAR, FONTTEST_DIR)
    with open(FONTTEST_META) as f:
        ft_meta = json.load(f)

    @torch.no_grad()
    def font_eval(mdl, cfg, tag):
        mdl.eval()
        pre = ImagePreprocessor(cfg)
        out = {}
        for split, items in ft_meta.items():
            preds = []
            for k in range(0, len(items), 64):
                chunk = items[k:k + 64]
                tens = [pre(Image.open(os.path.join(FONTTEST_DIR, it['file'])).convert('L')) for it in chunk]
                images, mask = collate_images(tens, cfg)
                with autocast('cuda'):
                    seqs = greedy_batch(mdl, images.to(DEVICE), mask.to(DEVICE), SOS, EOS, MAX_GEN_LEN)
                preds += [tokenizer.decode(s) for s in seqs]
            ids = [it['id'] for it in items]
            gl = [golds[i] for i in ids]
            tm, rm = text_metrics(gl, preds), render_metrics(gl, preds, ids, RENDER_DIR, f'ft_{tag}')
            fonts = sorted({it['font'] for it in items})
            by_font = {fo: 100.0 * np.mean([x for x, it in zip(rm['per_sample_img'], items)
                                             if it['font'] == fo and x is not None] or [0]) for fo in fonts}
            out[split] = {**strip(tm), **strip(rm), 'img_by_font': by_font,
                          'em_list': tm['per_sample_em'], 'img_list': rm['per_sample_img']}
            print(f"{tag:>8} | {split:16} | EM {tm['em']:.2f} % | image match {rm['img_match']:.2f} % | n={len(items)}")
            for it, g, pr in list(zip(items, gl, preds))[:2]:          # two examples, to see what the model reads
                print(f"           [{it['font']}] truth:  {g[:90]}\n           [{it['font']}] model:  {pr[:90]}")
        return out

    R = load_results()
    FT = R.get('font_test', {})
    if EXPERIMENT not in FT:
        FT[EXPERIMENT] = font_eval(model, MODEL_CFG, EXPERIMENT)
        R = save_results({'font_test': FT})
    g_path = os.path.join(SAVE_ROOT, 'experiments', 'baseline_seed42', 'model_best.pt')
    if EXPERIMENT != 'baseline' and 'baseline' not in FT and os.path.exists(g_path):
        g_cfg = {k: v for k, v in EXPERIMENTS['baseline'].items() if k not in ('ss_max', 'synth', 'aug')}
        g_model = Im2LatexModel(g_cfg, tokenizer.vocab_size, pretrained=False).to(DEVICE)
        g_model.load_state_dict(torch.load(g_path, map_location=DEVICE))
        FT['baseline'] = font_eval(g_model, g_cfg, 'baseline')
        R = save_results({'font_test': FT})
        del g_model
    if 'baseline' in FT and EXPERIMENT in FT and EXPERIMENT != 'baseline':
        for split in ft_meta:
            t = mcnemar_exact([x for x, y in zip(FT['baseline'][split]['img_list'], FT[EXPERIMENT][split]['img_list'])
                               if x is not None and y is not None],
                              [y for x, y in zip(FT['baseline'][split]['img_list'], FT[EXPERIMENT][split]['img_list'])
                               if x is not None and y is not None])
            FT[EXPERIMENT][split]['mcnemar_vs_baseline_img'] = t
            print(f"{split}: image match baseline {FT['baseline'][split]['img_match']:.2f} % -> {EXPERIMENT} "
                  f"{FT[EXPERIMENT][split]['img_match']:.2f} %  (McNemar p = {t[2]:.4f})")
        R = save_results({'font_test': FT})
else:
    print("The robustness test does not exist – cell 3b did not build it (check its output for the error).")

## 8. Ablation study (all experiments)

In [ ]:
# CELL 8: Ablation study – comparison table of all finished experiments
import glob
import matplotlib.pyplot as plt

DESC = {'baseline': 'Baseline model', 'no_aspect': '224×224 square instead of preserved aspect ratio',
        'block4': 'Block 4 (stride 32) instead of block 3', 'pos1d': '1D positions instead of 2D',
        'no_coverage': 'No coverage attention', 'no_ss': 'No scheduled sampling',
        'robust': 'Baseline + many fonts + augmentation'}
paths = sorted(glob.glob(os.path.join(SAVE_ROOT, 'experiments', '*', 'results.json')))
FT_ALL = {}                                   # the robustness test may be stored in another experiment's results
for path in paths:
    with open(path, encoding='utf-8') as f:
        FT_ALL.update(json.load(f).get('font_test', {}))
rows = []
for path in paths:
    with open(path, encoding='utf-8') as f:
        r = json.load(f)
    fg, M = r.get('full_greedy', {}), r.get('methods', {})
    if not fg:
        continue
    rows.append({'experiment': r['experiment'], 'seed': r['seed'], 'description': DESC.get(r['experiment'], ''),
                 'EM full test (%)': round(fg['em'], 2), 'Image match full test (%)': round(fg.get('img_match', float('nan')), 2),
                 'BLEU-4': round(fg['bleu'], 2), 'Compiles (%)': round(fg.get('compile', float('nan')), 2),
                 'Beam EM (%)': round(M.get('beam', {}).get('em', float('nan')), 2),
                 'Beam image match (%)': round(M.get('beam', {}).get('img_match', float('nan')), 2),
                 'Other fonts, image match (%)': round(FT_ALL.get(r['experiment'], {}).get('known_fonts', {}).get('img_match', float('nan')), 2),
                 'Unseen fonts, image match (%)': round(FT_ALL.get(r['experiment'], {}).get('unseen_fonts', {}).get('img_match', float('nan')), 2),
                 'min/epoch': round(np.mean(r['history']['epoch_minutes']), 1) if r.get('history') else None})
abl = pd.DataFrame(rows)
if len(abl):
    agg = abl.groupby(['experiment', 'description']).agg(['mean', 'std', 'count']).round(2) if abl['seed'].nunique() > 1 else abl
    display(agg.fillna('–'))
    fcols = ['Other fonts, image match (%)', 'Unseen fonts, image match (%)']
    missing = abl.loc[abl[fcols].isna().any(axis=1), 'experiment'].unique().tolist()
    if missing:
        print(f"'–' = not measured yet. The robustness test for {missing} is run by cell 7b: run that experiment "
              "again (Run all – training and evaluation are skipped, ≈ 15–25 min) or wait for 'robust', "
              "which also measures the baseline model.")
    abl.to_csv(os.path.join(SAVE_ROOT, 'ablation.csv'), index=False)
    order = [e for e in DESC if e in set(abl['experiment'])]
    means = abl.groupby('experiment')['Image match full test (%)'].mean().reindex(order)
    stds = abl.groupby('experiment')['Image match full test (%)'].std().reindex(order).fillna(0)
    plt.figure(figsize=(9, 3.6))
    bars = plt.bar([DESC[e] for e in order], means, yerr=stds, capsize=3,
                   color=['#D9822B' if e == 'baseline' else '#7FA7CF' for e in order])
    plt.bar_label(bars, labels=[f"{v:.1f}" for v in means], fontsize=9)
    plt.ylabel('Image match, full test (%)'); plt.xticks(rotation=20, ha='right'); plt.tight_layout()
    plt.savefig(os.path.join(SAVE_ROOT, 'ablation.png'), dpi=200); plt.show()
else:
    print("No finished experiments yet.")

## 9. Export the model for the web app

In [ ]:
# CELL 9: Export the model for the web app (+ upload to a Hugging Face Space)
HF_SPACE = ''          # e.g. 'your-username/merlin' (empty = save to Drive only)

EXPORT_EXPERIMENT, EXPORT_SEED = 'robust', 42      # which model goes to the web app (the robust model)

if (EXPERIMENT, SEED) != (EXPORT_EXPERIMENT, EXPORT_SEED):
    print(f"Export skipped: the web app uses EXPERIMENT = '{EXPORT_EXPERIMENT}' with SEED = {EXPORT_SEED}.")
else:
    R = load_results()
    fg, M = R.get('full_greedy', {}), R.get('methods', {})
    metrics = {
        'test_em_full': fg.get('em'), 'test_img_full': fg.get('img_match'), 'test_bleu_full': fg.get('bleu'),
        'test_compile_full': fg.get('compile'), 'n_test': fg.get('n'),
        'beam_em': M.get('beam', {}).get('em'), 'beam_img': M.get('beam', {}).get('img_match'),
        'early_exit_avg_layers': M.get('early_exit', {}).get('avg_layers'),
        'best_epoch': R.get('history', {}).get('best_epoch'), 'seed': SEED, 'experiment': EXPERIMENT,
        'font_img_known': R.get('font_test', {}).get(EXPERIMENT, {}).get('known_fonts', {}).get('img_match'),
        'font_img_unseen': R.get('font_test', {}).get(EXPERIMENT, {}).get('unseen_fonts', {}).get('img_match'),
    }
    WEB_DIR = os.path.join(SAVE_ROOT, 'web_model')
    model.load_state_dict(torch.load(CKPT_BEST, map_location=DEVICE))
    model.cpu()
    save_for_inference(model, tokenizer, WEB_DIR, {'metrics': metrics, 'max_gen_len': MAX_GEN_LEN})
    model.to(DEVICE)
    print("Model for the app saved to", WEB_DIR, os.listdir(WEB_DIR))

    if HF_SPACE:
        from google.colab import userdata
        from huggingface_hub import HfApi
        api = HfApi(token=userdata.get('HF_TOKEN'))           # Colab: key icon (left) -> Secrets -> HF_TOKEN
        api.upload_folder(repo_id=HF_SPACE, repo_type='space', folder_path=WEB_DIR, path_in_repo='model',
                          commit_message='Model from the MERLIN training notebook')
        print(f"Uploaded to https://huggingface.co/spaces/{HF_SPACE} – the Space will restart by itself.")